# Partie 1 — Exploration et préparation des données

In [11]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

### 1. Importation et examen du le jeu de données et de sa structure

In [12]:
df = pd.read_csv("hotels.csv")

print("Dimensions :", df.shape)
print(df.head())
df.info()                                   # types et valeurs non nulles
print(df.describe().T)                      # statistiques des colonnes numériques

Dimensions : (119390, 32)
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0                     0       2  .

### 2. Variables numériques / catégorielles + qualité des données

In [13]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

print("\nNumériques  :", len(num_cols), num_cols)
print("Catégorielles :", len(cat_cols), cat_cols)


Numériques  : 20 ['is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']
Catégorielles : 12 ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'reservation_status', 'reservation_status_date']


In [14]:
# --- Valeurs manquantes
manquants = df.isna().sum()
print("\nValeurs manquantes :\n", manquants[manquants > 0])
#   company 112 593 (~94 %), agent 16 340, country 488, children 4

# --- Doublons
print("Lignes dupliquées :", df.duplicated().sum())    # 31 994

# --- Valeurs aberrantes / incohérentes
print("adr négatif :", (df["adr"] < 0).sum())                          # 1
print("adr max :", df["adr"].max())                                    # 5400
total_clients = df["adults"] + df["children"].fillna(0) + df["babies"]
print("Réservations sans aucun client :", (total_clients == 0).sum())  # 180
print("Séjours de 0 nuit :",
      ((df["stays_in_weekend_nights"] + df["stays_in_week_nights"]) == 0).sum())  # 715
print("adults max :", df["adults"].max())                              # 55


Valeurs manquantes :
 children         4
country        488
agent        16340
company     112593
dtype: int64
Lignes dupliquées : 31994
adr négatif : 1
adr max : 5400.0
Réservations sans aucun client : 180
Séjours de 0 nuit : 715
adults max : 55


In [15]:
# --- Fuite de données (data leakage)
print(pd.crosstab(df["reservation_status"], df["is_canceled"]))
#   reservation_status reproduit exactement la cible -> à exclure

# --- Nettoyage minimal
df = df[df["adr"] >= 0]                     # retirer le prix négatif

df = df[total_clients.loc[df.index] > 0]    # retirer les réservations sans client

df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Inconnu")
df["agent"] = df["agent"].fillna(0).astype(int).astype(str)   # 0 = pas d'agent

df = df.drop(columns=["company"])           # trop de valeurs manquantes
print("Dimensions après nettoyage :", df.shape)

is_canceled             0      1
reservation_status              
Canceled                0  43017
Check-Out           75166      0
No-Show                 0   1207
Dimensions après nettoyage : (119209, 31)


### 3. Variables explicatives et variable cible

In [16]:
cible = "is_canceled"   # 1 = réservation annulée, 0 = non annulée

# Exclues : la cible + les colonnes connues seulement APRÈS l'issue
a_exclure = [cible, "reservation_status", "reservation_status_date"]

X = df.drop(columns=a_exclure)
y = df[cible]

print("\nVariables explicatives :", X.shape[1])
print(X.columns.tolist())
print("Répartition de la cible :\n", y.value_counts(normalize=True).round(3))


Variables explicatives : 28
['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']
Répartition de la cible :
 is_canceled
0    0.629
1    0.371
Name: proportion, dtype: float64


### 4. Découpage entraînement (80 %) / test (20 %)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,     # résultats reproductibles
    stratify=y           # même proportion d'annulations dans les 2 ensembles
)

print("\nX_train :", X_train.shape, "| X_test :", X_test.shape)
print("Taux d'annulation train :", round(y_train.mean(), 3),
      "| test :", round(y_test.mean(), 3))


X_train : (95367, 28) | X_test : (23842, 28)
Taux d'annulation train : 0.371 | test : 0.371


# Partie 2 — Construction des pipelines de prétraitement

### 1. Définir les traitements selon le type de variable

In [18]:
# a) Numériques très asymétriques (longue queue à droite)
#    -> imputation médiane + log(1+x) + standardisation
num_asymetriques = ["lead_time", "days_in_waiting_list"]

# b) Autres numériques (comptages, composantes de date, prix)
#    -> imputation médiane + standardisation
#    adr n'est PAS passé au log : ses nombreux tarifs à 0 créeraient
#    une forte asymétrie à gauche (skewness -3,75 testée au lieu de 1,03)
num_standard = [
    "adr",
    "arrival_date_year", "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies",
    "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "required_car_parking_spaces", "total_of_special_requests",
]

# c) Binaire déjà codée 0/1 -> aucune transformation
binaires = ["is_repeated_guest"]

# d) Catégorielles à peu de modalités -> imputation mode + one-hot
cat_faible = [
    "hotel", "arrival_date_month", "meal", "market_segment",
    "distribution_channel", "reserved_room_type", "assigned_room_type",
    "deposit_type", "customer_type",
]

# e) Catégorielles à beaucoup de modalités 
#    -> imputation "Inconnu" + one-hot avec regroupement des modalités rares
cat_forte = ["country", "agent"]

# Vérification : toutes les colonnes sont prises en charge, une seule fois
toutes = num_asymetriques + num_standard + binaires + cat_faible + cat_forte
assert sorted(toutes) == sorted(X_train.columns), "Colonne oubliée ou en double"


### 2. Pipeline par type de traitement

In [19]:
pipe_num_asym = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler()),
])

pipe_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

pipe_cat_faible = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

pipe_cat_forte = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Inconnu")),
    ("onehot", OneHotEncoder(
        min_frequency=0.005,                    # < 0,5 % des lignes -> "infrequent"
        handle_unknown="infrequent_if_exist",   # modalité inconnue au test -> "infrequent"
        sparse_output=False,
    )),
])

### 3. ColumnTransformer : chaque pipeline sur ses colonnes

In [20]:
preprocesseur = ColumnTransformer(
    transformers=[
        ("num_asym", pipe_num_asym, num_asymetriques),
        ("num", pipe_num, num_standard),
        ("bin", "passthrough", binaires),
        ("cat_faible", pipe_cat_faible, cat_faible),
        ("cat_forte", pipe_cat_forte, cat_forte),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# fit sur le TRAIN uniquement (médianes, moyennes, écarts-types, modalités),
# puis simple transform sur le test -> pas de fuite d'information du test
X_train_prep = preprocesseur.fit_transform(X_train)
X_test_prep = preprocesseur.transform(X_test)

noms = preprocesseur.get_feature_names_out()
X_train_prep = pd.DataFrame(X_train_prep, columns=noms, index=X_train.index)
X_test_prep = pd.DataFrame(X_test_prep, columns=noms, index=X_test.index)


### 4. Résultats après prétraitement

In [21]:
print("=== Dimensions ===")
print(f"X_train : {X_train.shape} -> {X_train_prep.shape}")
print(f"X_test  : {X_test.shape} -> {X_test_prep.shape}")

print("\n=== Nombre de colonnes produites par bloc ===")
print(pd.Series([n.split("__")[0] for n in noms]).value_counts().to_string())

print("\n=== Valeurs manquantes restantes ===")
print("train :", int(X_train_prep.isna().sum().sum()),
      "| test :", int(X_test_prep.isna().sum().sum()))

print("\n=== Effet du log sur l'asymétrie (skewness, train) ===")
for c in num_asymetriques:
    print(f"{c:22s} avant : {X_train[c].skew():6.2f}   après : "
          f"{X_train_prep['num_asym__' + c].skew():6.2f}")

print("\n=== Standardisation : moyenne ≈ 0, écart-type ≈ 1 (train) ===")
cols_num = [n for n in noms if n.startswith(("num__", "num_asym__"))]
print(X_train_prep[cols_num].agg(["mean", "std"]).T.round(3).head(6).to_string())

print("\n=== Modalités retenues pour country / agent ===")
ohe_forte = preprocesseur.named_transformers_["cat_forte"].named_steps["onehot"]
for var, cats, rares in zip(cat_forte, ohe_forte.categories_, ohe_forte.infrequent_categories_):
    n_rares = 0 if rares is None else len(rares)
    print(f"{var:8s} : {len(cats)} modalités -> {len(cats) - n_rares} gardées "
          f"+ 1 colonne 'infrequent' ({n_rares} regroupées)")

print("\n=== Aperçu de X_train prétraité ===")
print(X_train_prep.iloc[:5, :8].round(2).to_string())

=== Dimensions ===
X_train : (95367, 28) -> (95367, 120)
X_test  : (23842, 28) -> (23842, 120)

=== Nombre de colonnes produites par bloc ===
cat_faible    59
cat_forte     44
num           14
num_asym       2
bin            1

=== Valeurs manquantes restantes ===
train : 0 | test : 0

=== Effet du log sur l'asymétrie (skewness, train) ===
lead_time              avant :   1.35   après :  -0.87
days_in_waiting_list   avant :  11.87   après :   5.72

=== Standardisation : moyenne ≈ 0, écart-type ≈ 1 (train) ===
                                mean  std
num_asym__lead_time             -0.0  1.0
num_asym__days_in_waiting_list  -0.0  1.0
num__adr                        -0.0  1.0
num__arrival_date_year           0.0  1.0
num__arrival_date_week_number    0.0  1.0
num__arrival_date_day_of_month   0.0  1.0

=== Modalités retenues pour country / agent ===
country  : 171 modalités -> 20 gardées + 1 colonne 'infrequent' (151 regroupées)
agent    : 323 modalités -> 22 gardées + 1 colonne 'infrequen